# Comparative Study of Stochastic Models for Option Pricing in the Indian Stock Market
## Project Report — IIT Roorkee | Under Prof. Chaman Kumar

**Models:** Black-Scholes (GBM) → Merton Jump-Diffusion → Heston Stochastic Volatility → Bates (Heston + Jumps)

**Data:** NIFTY 50 daily closing prices (Jan 2018 – Dec 2024)

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import norm, probplot, gaussian_kde
from scipy.optimize import minimize
from scipy.special import factorial
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (14, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight'
})

FIGURES_DIR = 'figures/'
print('Setup complete.')

---
## 1. Data Acquisition & Exploratory Analysis

In [ ]:
ticker = '^NSEI'
data = yf.download(ticker, start='2018-01-01', end='2024-12-31')
prices = data['Close'].dropna()
if isinstance(prices, pd.DataFrame):
    prices = prices.iloc[:, 0]
log_returns = np.log(prices / prices.shift(1)).dropna()

print(f'Period: {prices.index[0].date()} to {prices.index[-1].date()}')
print(f'Total trading days: {len(prices)}')
print(f'Total log-return observations: {len(log_returns)}')
print(f'\nReturn statistics:')
print(f'  Mean daily return: {log_returns.mean():.6f}')
print(f'  Std daily return:  {log_returns.std():.6f}')
print(f'  Skewness:          {log_returns.skew():.4f}')
print(f'  Excess Kurtosis:   {log_returns.kurtosis():.4f}')
print(f'  Min daily return:  {log_returns.min():.6f}')
print(f'  Max daily return:  {log_returns.max():.6f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# NIFTY 50 Price History
ax = axes[0, 0]
ax.plot(prices.index, prices.values, color='#1B3A6B', linewidth=0.8)
events = {
    '2018-09-21': 'IL&FS Crisis',
    '2020-03-23': 'COVID Crash',
    '2022-02-24': 'Russia-Ukraine',
    '2024-06-04': 'Election Surprise'
}
for date_str, label in events.items():
    date = pd.Timestamp(date_str)
    if date in prices.index or True:
        ax.axvline(date, color='red', alpha=0.5, linestyle='--', linewidth=0.8)
        ax.text(date, ax.get_ylim()[1]*0.95, label, rotation=45, fontsize=7, ha='right', color='red')
ax.set_title('NIFTY 50 Price History (2018-2024) with Key Events')
ax.set_ylabel('Price (₹)')
ax.grid(True, alpha=0.3)

# Daily Log Returns
ax = axes[0, 1]
ax.plot(log_returns.index, log_returns.values, color='#C8521A', linewidth=0.3, alpha=0.8)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('Daily Log Returns')
ax.set_ylabel('Log Return')
ax.grid(True, alpha=0.3)

# Return Distribution with Normal Overlay
ax = axes[1, 0]
r_vals = log_returns.values
ax.hist(r_vals, bins=100, density=True, color='#1B3A6B', alpha=0.6, label='Empirical')
x_range = np.linspace(r_vals.min(), r_vals.max(), 500)
ax.plot(x_range, norm.pdf(x_range, r_vals.mean(), r_vals.std()), 
        'r-', linewidth=2, label=f'Normal fit (σ={r_vals.std():.4f})')
ax.set_title('Return Distribution vs Normal')
ax.set_xlabel('Log Return')
ax.set_ylabel('Density')
ax.legend()
ax.grid(True, alpha=0.3)

# Rolling 21-day Realized Volatility
ax = axes[1, 1]
rolling_vol = log_returns.rolling(21).std() * np.sqrt(252) * 100
ax.plot(rolling_vol.index, rolling_vol.values, color='#8B4513', linewidth=0.8)
ax.axhline(rolling_vol.mean(), color='green', linestyle='--', label=f'Mean: {rolling_vol.mean():.1f}%')
for date_str, label in events.items():
    ax.axvline(pd.Timestamp(date_str), color='red', alpha=0.4, linestyle='--', linewidth=0.8)
ax.set_title('21-Day Rolling Realized Volatility (Annualized)')
ax.set_ylabel('Volatility (%)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}01_data_overview.png')
plt.show()

---
## 2. Black-Scholes Model
### 2.1 MLE Parameter Estimation

Under GBM: $dS(t) = \mu S(t)dt + \sigma S(t)dW(t)$

Log-returns: $r_t = \ln(S_t/S_{t-1}) \sim \mathcal{N}\left((\mu - \frac{\sigma^2}{2})\Delta t,\ \sigma^2 \Delta t\right)$

Log-likelihood: $\ell(\mu, \sigma) = -\frac{n}{2}\ln(2\pi) - \frac{n}{2}\ln(\sigma^2\Delta t) - \sum_i \frac{(r_i - \tilde{\mu}\Delta t)^2}{2\sigma^2\Delta t}$

In [ ]:
r = log_returns.values
dt = 1/252

def neg_log_likelihood_bs(params, returns, dt=1/252):
    mu, sigma = params
    if sigma <= 0:
        return 1e10
    mean = (mu - 0.5 * sigma**2) * dt
    var = sigma**2 * dt
    ll = np.sum(norm.logpdf(returns, loc=mean, scale=np.sqrt(var)))
    return -ll

mu0 = r.mean() / dt + 0.5 * (r.std() / np.sqrt(dt))**2
sigma0 = r.std() / np.sqrt(dt)

result_bs = minimize(neg_log_likelihood_bs, x0=[mu0, sigma0],
                     args=(r, dt), method='Nelder-Mead',
                     options={'xatol': 1e-8, 'fatol': 1e-8})

mu_mle, sigma_mle = result_bs.x
ll_bs = -result_bs.fun

print('=== Black-Scholes MLE Parameters ===')
print(f'  μ (annualized drift):     {mu_mle:.6f} ({mu_mle*100:.2f}%)')
print(f'  σ (annualized volatility): {sigma_mle:.6f} ({sigma_mle*100:.2f}%)')
print(f'  Log-likelihood:            {ll_bs:.2f}')
print(f'  AIC:                       {2*2 - 2*ll_bs:.2f}')
print(f'  BIC:                       {2*np.log(len(r)) - 2*ll_bs:.2f}')

### 2.2 Black-Scholes Closed-Form Pricing

$C = S_0 N(d_1) - K e^{-r_f T} N(d_2)$

$d_1 = \frac{\ln(S_0/K) + (r_f + \sigma^2/2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$

In [ ]:
def black_scholes(S, K, T, r_f, sigma, option_type='call'):
    d1 = (np.log(S / K) + (r_f + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == 'call':
        price = S * norm.cdf(d1) - K * np.exp(-r_f * T) * norm.cdf(d2)
    else:
        price = K * np.exp(-r_f * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    return price, d1, d2

S0 = float(prices.iloc[-1])
K = round(S0 / 100) * 100
T = 30 / 252
r_f = 0.065

call_bs, d1, d2 = black_scholes(S0, K, T, r_f, sigma_mle, 'call')
put_bs, _, _ = black_scholes(S0, K, T, r_f, sigma_mle, 'put')

parity_lhs = call_bs - put_bs
parity_rhs = S0 - K * np.exp(-r_f * T)

print('=== Black-Scholes Closed-Form Prices ===')
print(f'  S₀ = ₹{S0:.2f},  K = ₹{K:.2f},  T = 30 days,  r_f = 6.5%')
print(f'  σ_MLE = {sigma_mle:.4f}')
print(f'  d₁ = {d1:.4f},  d₂ = {d2:.4f}')
print(f'  BS Call: ₹{call_bs:.2f}')
print(f'  BS Put:  ₹{put_bs:.2f}')
print(f'  Put-Call Parity check: LHS={parity_lhs:.4f}, RHS={parity_rhs:.4f}, Error={abs(parity_lhs-parity_rhs):.2e}')

### 2.3 Monte Carlo Simulation under Black-Scholes

Under $\mathbb{Q}$: $\ln S_{k+1} = \ln S_k + (r_f - \sigma^2/2)\Delta t + \sigma\sqrt{\Delta t} \cdot Z_k$

$\hat{C}_{MC} = e^{-r_f T} \cdot \frac{1}{N}\sum_{i=1}^{N} \max(S_T^{(i)} - K, 0)$

In [ ]:
def bs_monte_carlo(S0, K, T, r_f, sigma, n_paths=100_000, n_steps=252, seed=42):
    np.random.seed(seed)
    dt_mc = T / n_steps
    Z = np.random.standard_normal((n_paths, n_steps))
    inc = (r_f - 0.5 * sigma**2) * dt_mc + sigma * np.sqrt(dt_mc) * Z
    log_S = np.log(S0) + np.cumsum(inc, axis=1)
    S_T = np.exp(log_S[:, -1])
    payoff_call = np.maximum(S_T - K, 0)
    payoff_put = np.maximum(K - S_T, 0)
    discount = np.exp(-r_f * T)
    mc_call = discount * payoff_call.mean()
    mc_put = discount * payoff_put.mean()
    se_call = discount * payoff_call.std() / np.sqrt(n_paths)
    se_put = discount * payoff_put.std() / np.sqrt(n_paths)
    return mc_call, mc_put, se_call, se_put, log_S

mc_call_bs, mc_put_bs, se_c, se_p, log_paths_bs = bs_monte_carlo(S0, K, T, r_f, sigma_mle)

print('=== Black-Scholes Monte Carlo (100,000 paths) ===')
print(f'  MC Call:  ₹{mc_call_bs:.2f} ± {1.96*se_c:.4f} (95% CI)')
print(f'  MC Put:   ₹{mc_put_bs:.2f} ± {1.96*se_p:.4f} (95% CI)')
print(f'  CF Call:  ₹{call_bs:.2f}')
print(f'  CF Put:   ₹{put_bs:.2f}')
print(f'  Call error: {abs(mc_call_bs - call_bs):.4f}')
print(f'  Put error:  {abs(mc_put_bs - put_bs):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sample GBM paths
ax = axes[0]
n_show = 15
t_grid = np.linspace(0, T, log_paths_bs.shape[1])
for i in range(n_show):
    ax.plot(t_grid * 252, np.exp(log_paths_bs[i, :]), linewidth=0.6, alpha=0.7)
ax.axhline(K, color='red', linestyle='--', label=f'Strike K=₹{K}')
ax.set_title('Sample GBM Price Paths (Black-Scholes)')
ax.set_xlabel('Trading Days')
ax.set_ylabel('Price (₹)')
ax.legend()
ax.grid(True, alpha=0.3)

# Terminal price distribution
ax = axes[1]
S_T_bs = np.exp(log_paths_bs[:, -1])
ax.hist(S_T_bs, bins=150, density=True, color='#1B3A6B', alpha=0.6)
ax.axvline(K, color='red', linestyle='--', linewidth=2, label=f'Strike K=₹{K}')
ax.axvline(S0, color='green', linestyle='--', linewidth=2, label=f'S₀=₹{S0:.0f}')
ax.set_title('BS Terminal Price Distribution S(T)')
ax.set_xlabel('Price (₹)')
ax.set_ylabel('Density')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}02_bs_paths_and_terminal.png')
plt.show()

---
## 3. Merton Jump-Diffusion Model

### 3.1 MLE Parameter Estimation

SDE: $dS(t) = \mu S(t)dt + \sigma S(t)dW(t) + S(t^-)(e^J - 1)dN(t)$

where $N(t) \sim \text{Poisson}(\lambda)$, $J \sim \mathcal{N}(\mu_J, \sigma_J^2)$

Marginal density:
$f(r) = \sum_{k=0}^{\infty} \frac{e^{-\lambda\Delta t}(\lambda\Delta t)^k}{k!} \cdot \phi\left(r;\ \tilde{\mu}\Delta t + k\mu_J,\ \sigma^2\Delta t + k\sigma_J^2\right)$

In [ ]:
def mjd_log_likelihood(params, returns, dt=1/252, n_terms=20):
    mu, sigma, lam, mu_j, sigma_j = params
    if sigma <= 0 or lam < 0 or sigma_j <= 0:
        return 1e10
    n = len(returns)
    lam_dt = lam * dt
    
    log_poisson = np.array([-lam_dt + k * np.log(lam_dt + 1e-300) - np.sum(np.log(np.arange(1, k+1))) 
                           for k in range(n_terms)])
    
    ll = 0.0
    mu_tilde = mu - 0.5 * sigma**2
    for ri in returns:
        log_terms = np.zeros(n_terms)
        for k in range(n_terms):
            m_k = mu_tilde * dt + k * mu_j
            v_k = sigma**2 * dt + k * sigma_j**2
            if v_k <= 0:
                log_terms[k] = -1e10
                continue
            log_terms[k] = log_poisson[k] + norm.logpdf(ri, loc=m_k, scale=np.sqrt(v_k))
        max_log = np.max(log_terms)
        ll += max_log + np.log(np.sum(np.exp(log_terms - max_log)))
    return -ll

x0 = [mu_mle, sigma_mle * 0.8, 5.0, -0.02, 0.05]
bounds = [(None, None), (1e-4, None), (0, 50), (-0.5, 0.5), (1e-4, 1.0)]

res_mjd = minimize(mjd_log_likelihood, x0=x0, args=(r, 1/252, 15),
                   method='L-BFGS-B', bounds=bounds,
                   options={'maxiter': 500})

mu_m, sig_m, lam_m, muj_m, sigj_m = res_mjd.x
ll_mjd = -res_mjd.fun

print('=== Merton Jump-Diffusion MLE Parameters ===')
print(f'  μ (drift):            {mu_m:.6f} ({mu_m*100:.2f}%)')
print(f'  σ (diffusion vol):    {sig_m:.6f} ({sig_m*100:.2f}%)')
print(f'  λ (jump intensity):   {lam_m:.4f} jumps/year')
print(f'  μ_J (mean jump size): {muj_m:.6f}')
print(f'  σ_J (jump vol):       {sigj_m:.6f}')
print(f'  Log-likelihood:       {ll_mjd:.2f}')
print(f'  AIC:                  {2*5 - 2*ll_mjd:.2f}')
print(f'  BIC:                  {5*np.log(len(r)) - 2*ll_mjd:.2f}')

In [ ]:
# Likelihood Ratio Test: BS nested in MJD (when lambda -> 0)
from scipy.stats import chi2

LRT = 2 * (ll_mjd - ll_bs)
p_value = 1 - chi2.cdf(LRT, df=3)

print('=== Likelihood Ratio Test: BS vs MJD ===')
print(f'  ℓ(BS)  = {ll_bs:.2f}')
print(f'  ℓ(MJD) = {ll_mjd:.2f}')
print(f'  LRT statistic = 2[ℓ(MJD) - ℓ(BS)] = {LRT:.4f}')
print(f'  Degrees of freedom: 3 (λ, μ_J, σ_J)')
print(f'  p-value = {p_value:.2e}')
if p_value < 0.01:
    print('  → Jumps are HIGHLY SIGNIFICANT at 1% level. MJD is strongly preferred.')
elif p_value < 0.05:
    print('  → Jumps are significant at 5% level.')
else:
    print('  → Jumps are NOT significant.')

print('\n=== Information Criteria Comparison ===')
aic_bs = 2*2 - 2*ll_bs
aic_mjd = 2*5 - 2*ll_mjd
bic_bs = 2*np.log(len(r)) - 2*ll_bs
bic_mjd = 5*np.log(len(r)) - 2*ll_mjd
print(f'  AIC(BS)  = {aic_bs:.2f},    AIC(MJD) = {aic_mjd:.2f}  →  ΔAIC = {aic_bs - aic_mjd:.2f}')
print(f'  BIC(BS)  = {bic_bs:.2f},    BIC(MJD) = {bic_mjd:.2f}  →  ΔBIC = {bic_bs - bic_mjd:.2f}')

### 3.2 Monte Carlo Pricing under MJD

In [ ]:
def mjd_monte_carlo(S0, K, T, r_f, sigma, lam, mu_j, sigma_j,
                    n_paths=100_000, n_steps=252, seed=42, return_paths=False):
    np.random.seed(seed)
    dt_mc = T / n_steps
    kappa = np.exp(mu_j + 0.5 * sigma_j**2) - 1
    mu_rn = r_f - 0.5 * sigma**2 - lam * kappa
    
    log_S = np.full(n_paths, np.log(S0))
    if return_paths:
        all_log_S = np.zeros((n_paths, n_steps))
    
    for step in range(n_steps):
        Z = np.random.standard_normal(n_paths)
        N_jumps = np.random.poisson(lam * dt_mc, n_paths)
        J = np.where(N_jumps > 0,
                     np.array([np.random.normal(mu_j, sigma_j, int(n)).sum() if n > 0 else 0.0 
                              for n in N_jumps]),
                     0.0)
        log_S += mu_rn * dt_mc + sigma * np.sqrt(dt_mc) * Z + J
        if return_paths:
            all_log_S[:, step] = log_S
    
    S_T = np.exp(log_S)
    payoff_call = np.maximum(S_T - K, 0)
    payoff_put = np.maximum(K - S_T, 0)
    discount = np.exp(-r_f * T)
    
    result = (discount * payoff_call.mean(), discount * payoff_put.mean(),
              discount * payoff_call.std() / np.sqrt(n_paths),
              discount * payoff_put.std() / np.sqrt(n_paths))
    if return_paths:
        return result + (all_log_S,)
    return result

mjd_call, mjd_put, se_jc, se_jp, log_paths_mjd = mjd_monte_carlo(
    S0, K, T, r_f, sig_m, lam_m, muj_m, sigj_m, return_paths=True)

print('=== MJD Monte Carlo Prices (100,000 paths) ===')
print(f'  MJD MC Call: ₹{mjd_call:.2f} ± {1.96*se_jc:.4f}')
print(f'  MJD MC Put:  ₹{mjd_put:.2f} ± {1.96*se_jp:.4f}')
print(f'  BS CF Call:  ₹{call_bs:.2f}')
print(f'  BS CF Put:   ₹{put_bs:.2f}')
print(f'  Difference (Call): ₹{mjd_call - call_bs:.2f}')
print(f'  Difference (Put):  ₹{mjd_put - put_bs:.2f}')

---
## 4. Visual Comparisons: BS vs MJD

In [ ]:
# Figure: GBM vs MJD Sample Paths Side-by-Side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

np.random.seed(42)
n_demo = 8
n_steps_demo = int(T * 252)
dt_demo = T / n_steps_demo
t_demo = np.linspace(0, T * 252, n_steps_demo)

# GBM paths
ax = axes[0]
for i in range(n_demo):
    Z = np.random.standard_normal(n_steps_demo)
    log_s = np.log(S0) + np.cumsum((r_f - 0.5*sigma_mle**2)*dt_demo + sigma_mle*np.sqrt(dt_demo)*Z)
    ax.plot(t_demo, np.exp(log_s), linewidth=1, alpha=0.8)
ax.axhline(K, color='red', linestyle='--', label=f'Strike K=₹{K}')
ax.set_title('GBM Paths (Black-Scholes)', fontsize=13, fontweight='bold')
ax.set_xlabel('Trading Days')
ax.set_ylabel('Price (₹)')
ax.legend()
ax.grid(True, alpha=0.3)

# MJD paths
ax = axes[1]
kappa_demo = np.exp(muj_m + 0.5*sigj_m**2) - 1
mu_rn_demo = r_f - 0.5*sig_m**2 - lam_m*kappa_demo
for i in range(n_demo):
    Z = np.random.standard_normal(n_steps_demo)
    N_j = np.random.poisson(lam_m * dt_demo, n_steps_demo)
    J = np.array([np.random.normal(muj_m, sigj_m, int(n)).sum() if n > 0 else 0.0 for n in N_j])
    log_s = np.log(S0) + np.cumsum(mu_rn_demo*dt_demo + sig_m*np.sqrt(dt_demo)*Z + J)
    path = np.exp(log_s)
    ax.plot(t_demo, path, linewidth=1, alpha=0.8)
    jump_days = np.where(N_j > 0)[0]
    if len(jump_days) > 0:
        ax.scatter(t_demo[jump_days], path[jump_days], s=15, c='red', zorder=5, alpha=0.6)

ax.axhline(K, color='red', linestyle='--', label=f'Strike K=₹{K}')
ax.scatter([], [], s=15, c='red', label='Jump events')
ax.set_title('MJD Paths (Merton Jump-Diffusion)', fontsize=13, fontweight='bold')
ax.set_xlabel('Trading Days')
ax.set_ylabel('Price (₹)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}03_gbm_vs_mjd_paths.png')
plt.show()

In [ ]:
# Figure: Tail Behavior Comparison — BS vs MJD vs Empirical
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

n_sim = 200_000
np.random.seed(42)

bs_rets = np.random.normal((mu_mle - 0.5*sigma_mle**2)*dt, sigma_mle*np.sqrt(dt), n_sim)

kappa_sim = np.exp(muj_m + 0.5*sigj_m**2) - 1
Z_sim = np.random.standard_normal(n_sim)
N_j_sim = np.random.poisson(lam_m * dt, n_sim)
J_j_sim = np.array([np.random.normal(muj_m, sigj_m, int(n)).sum() if n > 0 else 0.0 for n in N_j_sim])
mjd_rets = (mu_m - 0.5*sig_m**2)*dt + sig_m*np.sqrt(dt)*Z_sim + J_j_sim

# Histogram comparison
ax = axes[0]
bins = np.linspace(-0.08, 0.08, 200)
ax.hist(r, bins=bins, density=True, alpha=0.5, color='gray', label='Empirical NIFTY')
ax.hist(bs_rets, bins=bins, density=True, alpha=0.4, color='#1B3A6B', label='BS (Gaussian)')
ax.hist(mjd_rets, bins=bins, density=True, alpha=0.4, color='#C8521A', label='MJD (Mixture)')
ax.set_title('Return Distributions', fontweight='bold')
ax.set_xlabel('Daily Log Return')
ax.set_ylabel('Density')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Left tail zoom
ax = axes[1]
bins_tail = np.linspace(-0.10, -0.02, 100)
ax.hist(r, bins=bins_tail, density=True, alpha=0.5, color='gray', label='Empirical NIFTY')
ax.hist(bs_rets, bins=bins_tail, density=True, alpha=0.4, color='#1B3A6B', label='BS')
ax.hist(mjd_rets, bins=bins_tail, density=True, alpha=0.4, color='#C8521A', label='MJD')
ax.axvline(np.percentile(r, 1), color='red', ls='--', lw=2, label='1% Empirical VaR')
ax.set_title('LEFT TAIL Zoom (Crash Risk)', fontweight='bold')
ax.set_xlabel('Daily Log Return')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# KDE overlay
ax = axes[2]
x_kde = np.linspace(-0.08, 0.08, 500)
kde_emp = gaussian_kde(r)
kde_bs = gaussian_kde(bs_rets)
kde_mjd = gaussian_kde(mjd_rets)
ax.plot(x_kde, kde_emp(x_kde), 'k-', linewidth=2.5, label='Empirical NIFTY')
ax.plot(x_kde, kde_bs(x_kde), '--', color='#1B3A6B', linewidth=2, label='BS')
ax.plot(x_kde, kde_mjd(x_kde), '--', color='#C8521A', linewidth=2, label='MJD')
ax.set_title('KDE Density Comparison', fontweight='bold')
ax.set_xlabel('Daily Log Return')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}04_tail_behavior_comparison.png')
plt.show()

In [ ]:
# Figure: QQ Plots — Empirical vs BS and MJD
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# QQ vs Normal (BS)
ax = axes[0]
probplot(r, dist='norm', plot=ax)
ax.set_title('QQ Plot: Empirical vs Normal (BS)', fontweight='bold')
ax.get_lines()[0].set_color('#1B3A6B')
ax.get_lines()[0].set_markersize(3)
ax.grid(True, alpha=0.3)

# QQ: Empirical vs MJD simulated
ax = axes[1]
emp_sorted = np.sort(r)
mjd_sorted = np.sort(mjd_rets[:len(r)])
ax.scatter(mjd_sorted, emp_sorted, s=3, color='#C8521A', alpha=0.5)
lims = [min(emp_sorted.min(), mjd_sorted.min()), max(emp_sorted.max(), mjd_sorted.max())]
ax.plot(lims, lims, 'k--', linewidth=1, label='45° line')
ax.set_xlabel('MJD Quantiles')
ax.set_ylabel('Empirical Quantiles')
ax.set_title('QQ Plot: Empirical vs MJD', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}05_qq_plots.png')
plt.show()

In [ ]:
# Figure: Log-Likelihood and Information Criteria Comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

models = ['Black-Scholes', 'Merton JD']
colors = ['#1B3A6B', '#C8521A']

# Log-Likelihood
ax = axes[0]
bars = ax.bar(models, [ll_bs, ll_mjd], color=colors, alpha=0.8, edgecolor='black')
for bar, val in zip(bars, [ll_bs, ll_mjd]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 10,
            f'{val:.1f}', ha='center', va='bottom', fontweight='bold')
ax.set_title('Log-Likelihood ℓ(θ̂)', fontweight='bold')
ax.set_ylabel('Log-Likelihood')
ax.grid(True, alpha=0.3, axis='y')

# AIC
ax = axes[1]
bars = ax.bar(models, [aic_bs, aic_mjd], color=colors, alpha=0.8, edgecolor='black')
for bar, val in zip(bars, [aic_bs, aic_mjd]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{val:.1f}', ha='center', va='bottom', fontweight='bold')
ax.set_title('AIC (lower is better)', fontweight='bold')
ax.set_ylabel('AIC')
ax.grid(True, alpha=0.3, axis='y')

# BIC
ax = axes[2]
bars = ax.bar(models, [bic_bs, bic_mjd], color=colors, alpha=0.8, edgecolor='black')
for bar, val in zip(bars, [bic_bs, bic_mjd]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{val:.1f}', ha='center', va='bottom', fontweight='bold')
ax.set_title('BIC (lower is better)', fontweight='bold')
ax.set_ylabel('BIC')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}06_model_comparison_criteria.png')
plt.show()

In [ ]:
# Figure: Implied Volatility Smile — BS IV backed out from MJD prices
strikes = np.linspace(S0 * 0.85, S0 * 1.15, 30)
iv_from_mjd = []

def bs_iv(market_price, S, K, T, r_f, option_type='call'):
    from scipy.optimize import brentq
    def objective(sigma):
        p, _, _ = black_scholes(S, K, T, r_f, sigma, option_type)
        return p - market_price
    try:
        return brentq(objective, 0.01, 3.0)
    except:
        return np.nan

for Ki in strikes:
    mjd_price, _, _, _ = mjd_monte_carlo(S0, Ki, T, r_f, sig_m, lam_m, muj_m, sigj_m,
                                          n_paths=50_000, n_steps=60, seed=42)
    iv = bs_iv(mjd_price, S0, Ki, T, r_f, 'call')
    iv_from_mjd.append(iv)

iv_from_mjd = np.array(iv_from_mjd)
moneyness = strikes / S0

fig, ax = plt.subplots(figsize=(12, 6))
valid = ~np.isnan(iv_from_mjd)
ax.plot(moneyness[valid], iv_from_mjd[valid] * 100, 'o-', color='#C8521A', 
        linewidth=2, markersize=5, label='BS IV from MJD prices')
ax.axhline(sigma_mle * 100, color='#1B3A6B', linestyle='--', linewidth=2, 
           label=f'BS constant σ = {sigma_mle*100:.1f}%')
ax.axvline(1.0, color='gray', linestyle=':', alpha=0.5, label='ATM (K/S₀ = 1)')
ax.set_xlabel('Moneyness (K/S₀)', fontsize=12)
ax.set_ylabel('Implied Volatility (%)', fontsize=12)
ax.set_title('Implied Volatility Smile: MJD generates skew that BS cannot produce', 
             fontweight='bold', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}07_implied_volatility_smile.png')
plt.show()

In [ ]:
# Figure: Parameter Sensitivity Heatmap — MJD option price as function of (λ, σ_J)
lambda_range = np.linspace(0.5, 30, 15)
sigj_range = np.linspace(0.01, 0.15, 15)
price_grid = np.zeros((len(sigj_range), len(lambda_range)))

for i, sj in enumerate(sigj_range):
    for j, lm in enumerate(lambda_range):
        p, _, _, _ = mjd_monte_carlo(S0, K, T, r_f, sig_m, lm, muj_m, sj,
                                      n_paths=20_000, n_steps=30, seed=42)
        price_grid[i, j] = p

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(price_grid, 
            xticklabels=[f'{l:.0f}' for l in lambda_range],
            yticklabels=[f'{s:.3f}' for s in sigj_range],
            annot=True, fmt='.0f', cmap='YlOrRd', ax=ax)
ax.set_xlabel('λ (Jump Intensity, jumps/year)', fontsize=12)
ax.set_ylabel('σ_J (Jump Volatility)', fontsize=12)
ax.set_title('MJD ATM Call Price (₹) — Sensitivity to Jump Parameters', 
             fontweight='bold', fontsize=13)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}08_parameter_sensitivity_heatmap.png')
plt.show()

In [ ]:
# Figure: Monte Carlo Convergence Analysis
path_counts = [100, 500, 1000, 2000, 5000, 10000, 20000, 50000, 100000, 200000, 500000]

mc_prices_bs = []
mc_ses_bs = []
mc_prices_mjd = []
mc_ses_mjd = []

for N in path_counts:
    c_bs, _, se_bs, _, _ = bs_monte_carlo(S0, K, T, r_f, sigma_mle, n_paths=N, seed=42)
    mc_prices_bs.append(c_bs)
    mc_ses_bs.append(se_bs)
    
    c_mjd, _, se_mjd, _ = mjd_monte_carlo(S0, K, T, r_f, sig_m, lam_m, muj_m, sigj_m,
                                            n_paths=N, n_steps=60, seed=42)
    mc_prices_mjd.append(c_mjd)
    mc_ses_mjd.append(se_mjd)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
ax.semilogx(path_counts, mc_prices_bs, 'o-', color='#1B3A6B', label='BS MC Price')
ax.fill_between(path_counts, 
                [p - 1.96*s for p, s in zip(mc_prices_bs, mc_ses_bs)],
                [p + 1.96*s for p, s in zip(mc_prices_bs, mc_ses_bs)],
                alpha=0.2, color='#1B3A6B')
ax.axhline(call_bs, color='red', linestyle='--', label=f'BS Closed-Form = ₹{call_bs:.2f}')
ax.set_title('BS MC Convergence', fontweight='bold')
ax.set_xlabel('Number of Paths (log scale)')
ax.set_ylabel('Call Price (₹)')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.loglog(path_counts, [1.96*s for s in mc_ses_bs], 'o-', color='#1B3A6B', label='BS MC SE')
ax.loglog(path_counts, [1.96*s for s in mc_ses_mjd], 's-', color='#C8521A', label='MJD MC SE')
N_ref = np.array(path_counts)
ax.loglog(N_ref, mc_ses_bs[0]*1.96 * np.sqrt(path_counts[0]) / np.sqrt(N_ref), 
          'k--', alpha=0.5, label='1/√N reference')
ax.set_title('MC Standard Error (95% CI width)', fontweight='bold')
ax.set_xlabel('Number of Paths')
ax.set_ylabel('95% CI Half-Width (₹)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}09_mc_convergence.png')
plt.show()

---
## 5. Market Event Analysis — Indian Market Stress Tests

In [ ]:
# Event Analysis: pricing errors at key Indian market events
event_dates = {
    'IL&FS Crisis\n(Sep 2018)': '2018-09-21',
    'COVID Crash\n(Mar 2020)': '2020-03-23',
    'Russia-Ukraine\n(Feb 2022)': '2022-02-24',
    'Election Surprise\n(Jun 2024)': '2024-06-04'
}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

event_returns = {}
for idx, (label, date_str) in enumerate(event_dates.items()):
    ax = axes[idx // 2, idx % 2]
    event_date = pd.Timestamp(date_str)
    
    window_start = event_date - pd.Timedelta(days=60)
    window_end = event_date + pd.Timedelta(days=60)
    mask = (prices.index >= window_start) & (prices.index <= window_end)
    window_prices = prices[mask]
    window_returns = log_returns[mask]
    
    ax.plot(window_prices.index, window_prices.values, color='#1B3A6B', linewidth=1.5)
    ax.axvline(event_date, color='red', linestyle='--', linewidth=2, alpha=0.8)
    
    nearest_idx = window_prices.index.get_indexer([event_date], method='nearest')[0]
    if nearest_idx >= 0 and nearest_idx < len(window_prices):
        event_price = window_prices.iloc[nearest_idx]
        ax.scatter([window_prices.index[nearest_idx]], [event_price], 
                   color='red', s=100, zorder=5)
        
        if nearest_idx > 0:
            ret_1d = np.log(window_prices.iloc[nearest_idx] / window_prices.iloc[nearest_idx-1])
            ax.text(0.05, 0.95, f'1-day return: {ret_1d*100:.1f}%', 
                    transform=ax.transAxes, fontsize=10, fontweight='bold',
                    color='red', va='top')
            event_returns[label] = ret_1d
    
    ax.set_title(label.replace('\n', ' '), fontweight='bold', fontsize=12)
    ax.set_ylabel('NIFTY 50')
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

plt.suptitle('NIFTY 50 Around Key Market Events (±60 days)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}10_event_analysis.png')
plt.show()

In [ ]:
# Figure: BS vs MJD — probability of extreme events
thresholds = np.linspace(-0.15, -0.01, 50)

prob_bs = norm.cdf(thresholds, loc=(mu_mle - 0.5*sigma_mle**2)*dt, scale=sigma_mle*np.sqrt(dt))

def mjd_cdf(x, mu, sigma, lam, mu_j, sigma_j, dt, n_terms=20):
    cdf_val = 0.0
    lam_dt = lam * dt
    for k in range(n_terms):
        m_k = (mu - 0.5*sigma**2)*dt + k*mu_j
        v_k = sigma**2*dt + k*sigma_j**2
        p_k = np.exp(-lam_dt) * lam_dt**k / np.math.factorial(k)
        cdf_val += p_k * norm.cdf(x, loc=m_k, scale=np.sqrt(v_k))
    return cdf_val

prob_mjd = np.array([mjd_cdf(t, mu_m, sig_m, lam_m, muj_m, sigj_m, dt) for t in thresholds])

emp_prob = np.array([np.mean(r <= t) for t in thresholds])

fig, ax = plt.subplots(figsize=(12, 6))
ax.semilogy(-thresholds*100, emp_prob, 'ko-', markersize=3, label='Empirical NIFTY', linewidth=2)
ax.semilogy(-thresholds*100, prob_bs, '--', color='#1B3A6B', linewidth=2, label='BS (Gaussian)')
ax.semilogy(-thresholds*100, prob_mjd, '--', color='#C8521A', linewidth=2, label='MJD')

ax.axvline(13, color='red', alpha=0.3, linewidth=8, label='COVID crash (13%)')
ax.axvline(8, color='orange', alpha=0.3, linewidth=8, label='Election surprise (8%)')

ax.set_xlabel('Loss Magnitude (% daily drop)', fontsize=12)
ax.set_ylabel('Probability (log scale)', fontsize=12)
ax.set_title('Tail Risk: Probability of Daily Loss Exceeding X%\nBS vastly underestimates crash probability',
             fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.invert_xaxis()

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}11_tail_risk_probability.png')
plt.show()

In [ ]:
# Figure: VaR Backtesting — 1% VaR under BS vs MJD
window = 252
var_bs_series = []
var_mjd_series = []
dates_var = []
actual_returns_var = []

for i in range(window, len(r)):
    window_r = r[i-window:i]
    mu_w = window_r.mean() / dt + 0.5 * (window_r.std()/np.sqrt(dt))**2
    sig_w = window_r.std() / np.sqrt(dt)
    
    var_bs_1pct = norm.ppf(0.01, loc=(mu_w - 0.5*sig_w**2)*dt, scale=sig_w*np.sqrt(dt))
    var_bs_series.append(var_bs_1pct)
    
    var_mjd_1pct = np.percentile(window_r, 1)
    var_mjd_series.append(var_mjd_1pct)
    
    dates_var.append(log_returns.index[i])
    actual_returns_var.append(r[i])

var_bs_series = np.array(var_bs_series)
var_mjd_series = np.array(var_mjd_series)
actual_returns_var = np.array(actual_returns_var)

breaches_bs = actual_returns_var < var_bs_series
breaches_mjd = actual_returns_var < var_mjd_series

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

ax = axes[0]
ax.plot(dates_var, actual_returns_var * 100, color='gray', alpha=0.4, linewidth=0.5, label='Actual returns')
ax.plot(dates_var, var_bs_series * 100, color='#1B3A6B', linewidth=1.5, label='BS 1% VaR')
ax.plot(dates_var, var_mjd_series * 100, color='#C8521A', linewidth=1.5, label='Empirical 1% VaR')
breach_idx_bs = np.where(breaches_bs)[0]
ax.scatter(np.array(dates_var)[breach_idx_bs], actual_returns_var[breach_idx_bs]*100, 
           c='red', s=15, zorder=5, label=f'BS breaches ({breaches_bs.sum()})')
ax.set_title('Value-at-Risk Backtesting (1% Daily VaR, 252-day rolling)', fontweight='bold')
ax.set_ylabel('Return (%)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
cum_breaches_bs = np.cumsum(breaches_bs)
cum_breaches_mjd = np.cumsum(breaches_mjd)
expected = np.arange(1, len(breaches_bs)+1) * 0.01
ax.plot(dates_var, cum_breaches_bs, color='#1B3A6B', linewidth=2, label=f'BS breaches (total: {breaches_bs.sum()})')
ax.plot(dates_var, cum_breaches_mjd, color='#C8521A', linewidth=2, label=f'Emp breaches (total: {breaches_mjd.sum()})')
ax.plot(dates_var, expected, 'k--', linewidth=1, label='Expected (1%)')
ax.fill_between(dates_var, expected - 1.96*np.sqrt(expected*(1-0.01)), 
                expected + 1.96*np.sqrt(expected*(1-0.01)), alpha=0.15, color='gray',
                label='95% Kupiec band')
ax.set_title('Cumulative VaR Breaches', fontweight='bold')
ax.set_ylabel('Cumulative Breaches')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}12_var_backtesting.png')
plt.show()

print(f'BS VaR breach rate:  {breaches_bs.mean()*100:.2f}% (expected: 1%)')
print(f'Emp VaR breach rate: {breaches_mjd.mean()*100:.2f}% (expected: 1%)')

---
## 6. Variance Reduction Techniques

In [ ]:
# Antithetic Variates and Control Variate MC
def bs_mc_antithetic(S0, K, T, r_f, sigma, n_paths=100_000, seed=42):
    np.random.seed(seed)
    n_half = n_paths // 2
    Z = np.random.standard_normal(n_half)
    
    S_T_pos = S0 * np.exp((r_f - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    S_T_neg = S0 * np.exp((r_f - 0.5*sigma**2)*T + sigma*np.sqrt(T)*(-Z))
    
    payoff_pos = np.maximum(S_T_pos - K, 0)
    payoff_neg = np.maximum(S_T_neg - K, 0)
    payoff_avg = (payoff_pos + payoff_neg) / 2
    
    discount = np.exp(-r_f * T)
    price = discount * payoff_avg.mean()
    se = discount * payoff_avg.std() / np.sqrt(n_half)
    return price, se

def bs_mc_control_variate(S0, K, T, r_f, sigma, n_paths=100_000, seed=42):
    np.random.seed(seed)
    Z = np.random.standard_normal(n_paths)
    S_T = S0 * np.exp((r_f - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    
    payoff = np.maximum(S_T - K, 0)
    control = S_T
    expected_control = S0 * np.exp(r_f * T)
    
    cov_pc = np.cov(payoff, control)[0, 1]
    var_c = np.var(control)
    beta = cov_pc / var_c
    
    adjusted_payoff = payoff - beta * (control - expected_control)
    discount = np.exp(-r_f * T)
    price = discount * adjusted_payoff.mean()
    se = discount * adjusted_payoff.std() / np.sqrt(n_paths)
    return price, se

naive_price, naive_se = call_bs, mc_ses_bs[-1] if mc_ses_bs else se_c
# Rerun naive
np.random.seed(42)
Z_naive = np.random.standard_normal(100_000)
S_T_naive = S0 * np.exp((r_f - 0.5*sigma_mle**2)*T + sigma_mle*np.sqrt(T)*Z_naive)
payoff_naive = np.maximum(S_T_naive - K, 0)
naive_price = np.exp(-r_f*T) * payoff_naive.mean()
naive_se = np.exp(-r_f*T) * payoff_naive.std() / np.sqrt(100_000)

anti_price, anti_se = bs_mc_antithetic(S0, K, T, r_f, sigma_mle)
cv_price, cv_se = bs_mc_control_variate(S0, K, T, r_f, sigma_mle)

print('=== Variance Reduction Comparison (100,000 paths) ===')
print(f'  Closed-form:       ₹{call_bs:.4f}')
print(f'  Naive MC:          ₹{naive_price:.4f}  SE={naive_se:.4f}')
print(f'  Antithetic MC:     ₹{anti_price:.4f}  SE={anti_se:.4f}  (reduction: {(1-anti_se/naive_se)*100:.1f}%)')
print(f'  Control Variate:   ₹{cv_price:.4f}  SE={cv_se:.4f}  (reduction: {(1-cv_se/naive_se)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(10, 6))
methods = ['Naive MC', 'Antithetic', 'Control Variate']
ses = [naive_se, anti_se, cv_se]
prices_vr = [naive_price, anti_price, cv_price]
colors_vr = ['#1B3A6B', '#2E8B57', '#C8521A']

ax.bar(methods, ses, color=colors_vr, alpha=0.8, edgecolor='black')
for i, (m, s) in enumerate(zip(methods, ses)):
    ax.text(i, s + 0.0005, f'SE={s:.4f}', ha='center', fontweight='bold', fontsize=11)
ax.axhline(0, color='black')
ax.set_title('Monte Carlo Standard Error — Variance Reduction Comparison', fontweight='bold')
ax.set_ylabel('Standard Error (₹)')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}13_variance_reduction.png')
plt.show()

---
## 7. Heston Stochastic Volatility Model (End-Term Proposal)

$dS(t) = r_f S(t)dt + \sqrt{v(t)} S(t) dW_S^{\mathbb{Q}}(t)$

$dv(t) = \kappa[\theta - v(t)]dt + \xi \sqrt{v(t)} dW_v^{\mathbb{Q}}(t)$

$\text{Corr}(dW_S, dW_v) = \rho\,dt$

Feller condition: $2\kappa\theta > \xi^2$ ensures $v(t) > 0$

In [ ]:
# Heston Monte Carlo with Euler-Maruyama
def heston_mc(S0, K, T, r_f, v0, kappa, theta, xi, rho,
              n_paths=50_000, n_steps=252, seed=42, return_paths=False):
    np.random.seed(seed)
    dt_h = T / n_steps
    
    S = np.full(n_paths, float(S0))
    v = np.full(n_paths, float(v0))
    
    if return_paths:
        S_paths = np.zeros((n_paths, n_steps))
        v_paths = np.zeros((n_paths, n_steps))
    
    for step in range(n_steps):
        Z1 = np.random.standard_normal(n_paths)
        Z2 = rho * Z1 + np.sqrt(1 - rho**2) * np.random.standard_normal(n_paths)
        
        v_pos = np.maximum(v, 0)
        S = S * np.exp((r_f - 0.5*v_pos)*dt_h + np.sqrt(v_pos*dt_h)*Z1)
        v = v + kappa*(theta - v)*dt_h + xi*np.sqrt(v_pos*dt_h)*Z2
        v = np.maximum(v, 0)
        
        if return_paths:
            S_paths[:, step] = S
            v_paths[:, step] = v
    
    payoff_call = np.maximum(S - K, 0)
    payoff_put = np.maximum(K - S, 0)
    discount = np.exp(-r_f * T)
    
    result = {
        'call': discount * payoff_call.mean(),
        'put': discount * payoff_put.mean(),
        'se_call': discount * payoff_call.std() / np.sqrt(n_paths),
        'se_put': discount * payoff_put.std() / np.sqrt(n_paths)
    }
    if return_paths:
        result['S_paths'] = S_paths
        result['v_paths'] = v_paths
    return result

# Realistic Heston parameters for NIFTY 50
v0_h = sigma_mle**2
kappa_h = 3.0
theta_h = sigma_mle**2
xi_h = 0.4
rho_h = -0.7

print(f'Feller condition: 2κθ = {2*kappa_h*theta_h:.4f} vs ξ² = {xi_h**2:.4f}', 
      '→ SATISFIED' if 2*kappa_h*theta_h > xi_h**2 else '→ VIOLATED')

heston_result = heston_mc(S0, K, T, r_f, v0_h, kappa_h, theta_h, xi_h, rho_h,
                          n_paths=50_000, return_paths=True)

print(f'\n=== Heston MC Prices ===')
print(f'  Heston Call: ₹{heston_result["call"]:.2f} ± {1.96*heston_result["se_call"]:.4f}')
print(f'  Heston Put:  ₹{heston_result["put"]:.2f} ± {1.96*heston_result["se_put"]:.4f}')
print(f'  BS Call:     ₹{call_bs:.2f}')
print(f'  MJD Call:    ₹{mjd_call:.2f}')

In [ ]:
# Figure: Heston — Price and Variance Paths
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

n_show = 10
t_heston = np.linspace(0, T*252, heston_result['S_paths'].shape[1])

ax = axes[0]
for i in range(n_show):
    ax.plot(t_heston, heston_result['S_paths'][i, :], linewidth=0.8, alpha=0.7)
ax.axhline(K, color='red', linestyle='--', label=f'Strike K=₹{K}')
ax.set_title('Heston Model: Sample Price Paths S(t)', fontweight='bold')
ax.set_ylabel('Price (₹)')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
for i in range(n_show):
    ax.plot(t_heston, np.sqrt(heston_result['v_paths'][i, :]) * 100, linewidth=0.8, alpha=0.7)
ax.axhline(np.sqrt(theta_h) * 100, color='red', linestyle='--', 
           label=f'Long-run √θ = {np.sqrt(theta_h)*100:.1f}%')
ax.set_title('Heston Model: Stochastic Volatility √v(t)', fontweight='bold')
ax.set_xlabel('Trading Days')
ax.set_ylabel('Volatility (%)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}14_heston_paths.png')
plt.show()

In [ ]:
# Figure: Heston Implied Volatility Surface
maturities = [7, 14, 30, 60, 90, 180]
moneyness_range = np.linspace(0.85, 1.15, 20)

iv_surface = np.zeros((len(maturities), len(moneyness_range)))

for i, T_mat in enumerate(maturities):
    T_y = T_mat / 252
    for j, m in enumerate(moneyness_range):
        K_i = S0 * m
        h_res = heston_mc(S0, K_i, T_y, r_f, v0_h, kappa_h, theta_h, xi_h, rho_h,
                          n_paths=20_000, n_steps=max(T_mat, 30), seed=42)
        iv = bs_iv(h_res['call'], S0, K_i, T_y, r_f, 'call')
        iv_surface[i, j] = iv * 100 if not np.isnan(iv) else np.nan

fig, ax = plt.subplots(figsize=(14, 8))
X, Y = np.meshgrid(moneyness_range, maturities)
cs = ax.contourf(X, Y, iv_surface, levels=20, cmap='RdYlBu_r')
plt.colorbar(cs, label='Implied Volatility (%)')
ax.set_xlabel('Moneyness (K/S₀)', fontsize=12)
ax.set_ylabel('Maturity (trading days)', fontsize=12)
ax.set_title('Heston Implied Volatility Surface\nStochastic vol produces realistic smile & term structure',
             fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}15_heston_iv_surface.png')
plt.show()

In [ ]:
# Figure: Euler vs Tamed Euler for CIR Variance Process
# Connection to Dr. Chaman Kumar's research (Dareiotis, Kumar & Sabanis, 2016)

def simulate_cir_euler(v0, kappa, theta, xi, T, n_steps, seed=42):
    np.random.seed(seed)
    dt_c = T / n_steps
    v = np.zeros(n_steps + 1)
    v[0] = v0
    for i in range(n_steps):
        Z = np.random.standard_normal()
        v[i+1] = v[i] + kappa*(theta - v[i])*dt_c + xi*np.sqrt(abs(v[i])*dt_c)*Z
    return v

def simulate_cir_tamed_euler(v0, kappa, theta, xi, T, n_steps, seed=42):
    np.random.seed(seed)
    dt_c = T / n_steps
    v = np.zeros(n_steps + 1)
    v[0] = v0
    for i in range(n_steps):
        Z = np.random.standard_normal()
        v_pos = max(v[i], 0)
        drift = kappa*(theta - v_pos)
        diffusion = xi*np.sqrt(v_pos)
        taming_factor = 1.0 / (1.0 + dt_c * (abs(drift) + diffusion**2))
        v[i+1] = v[i] + taming_factor * drift * dt_c + taming_factor * diffusion * np.sqrt(dt_c) * Z
        v[i+1] = max(v[i+1], 0)
    return v

T_cir = 1.0
n_steps_cir = 500
v0_cir = 0.04
kappa_cir = 2.0
theta_cir = 0.04
xi_cir = 0.8  # Deliberately high vol-of-vol to stress test

t_cir = np.linspace(0, T_cir, n_steps_cir + 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
for seed_i in range(5):
    v_euler = simulate_cir_euler(v0_cir, kappa_cir, theta_cir, xi_cir, T_cir, n_steps_cir, seed=seed_i)
    v_tamed = simulate_cir_tamed_euler(v0_cir, kappa_cir, theta_cir, xi_cir, T_cir, n_steps_cir, seed=seed_i)
    ax.plot(t_cir, v_euler, color='#1B3A6B', alpha=0.5, linewidth=0.8)
    ax.plot(t_cir, v_tamed, color='#C8521A', alpha=0.5, linewidth=0.8)

ax.plot([], [], color='#1B3A6B', label='Standard Euler')
ax.plot([], [], color='#C8521A', label='Tamed Euler (DKS16)')
ax.axhline(0, color='black', linewidth=0.5)
ax.axhline(theta_cir, color='green', linestyle='--', alpha=0.5, label=f'θ = {theta_cir}')
ax.set_title('CIR Variance Process: Standard vs Tamed Euler\n(ξ = 0.8, Feller condition violated)', 
             fontweight='bold')
ax.set_xlabel('Time (years)')
ax.set_ylabel('v(t)')
ax.legend()
ax.grid(True, alpha=0.3)

# Convergence analysis
ax = axes[1]
step_sizes = [50, 100, 200, 500, 1000, 2000, 5000]
errors_euler = []
errors_tamed = []
n_mc = 5000

# Reference solution with very fine grid
ref_vals = []
for seed_i in range(n_mc):
    v_ref = simulate_cir_tamed_euler(v0_cir, kappa_cir, theta_cir, xi_cir, T_cir, 10000, seed=seed_i)
    ref_vals.append(v_ref[-1])
ref_mean = np.mean(ref_vals)

for ns in step_sizes:
    euler_vals = []
    tamed_vals = []
    for seed_i in range(n_mc):
        v_e = simulate_cir_euler(v0_cir, kappa_cir, theta_cir, xi_cir, T_cir, ns, seed=seed_i)
        v_t = simulate_cir_tamed_euler(v0_cir, kappa_cir, theta_cir, xi_cir, T_cir, ns, seed=seed_i)
        euler_vals.append(v_e[-1])
        tamed_vals.append(v_t[-1])
    errors_euler.append(abs(np.mean(euler_vals) - ref_mean))
    errors_tamed.append(abs(np.mean(tamed_vals) - ref_mean))

dt_vals = [T_cir / ns for ns in step_sizes]
ax.loglog(dt_vals, errors_euler, 'o-', color='#1B3A6B', linewidth=2, label='Standard Euler')
ax.loglog(dt_vals, errors_tamed, 's-', color='#C8521A', linewidth=2, label='Tamed Euler (DKS16)')
dt_ref = np.array(dt_vals)
ax.loglog(dt_ref, 0.01*np.sqrt(dt_ref), 'k--', alpha=0.5, label='O(√Δt) reference')
ax.set_title('Weak Convergence: E[v(T)] Error vs Step Size', fontweight='bold')
ax.set_xlabel('Δt')
ax.set_ylabel('|E[v̂(T)] - E[v(T)]|')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}16_tamed_euler_convergence.png')
plt.show()

---
## 8. Option Pricing Across Strike Range — Full Comparison

In [ ]:
# Price calls across strikes for all models
strike_range = np.linspace(S0 * 0.85, S0 * 1.15, 25)
bs_prices = []
mjd_prices = []
heston_prices = []

for Ki in strike_range:
    # BS closed-form
    p_bs, _, _ = black_scholes(S0, Ki, T, r_f, sigma_mle, 'call')
    bs_prices.append(p_bs)
    
    # MJD MC
    p_mjd, _, _, _ = mjd_monte_carlo(S0, Ki, T, r_f, sig_m, lam_m, muj_m, sigj_m,
                                      n_paths=30_000, n_steps=60, seed=42)
    mjd_prices.append(p_mjd)
    
    # Heston MC
    h_res = heston_mc(S0, Ki, T/252*252, r_f, v0_h, kappa_h, theta_h, xi_h, rho_h,
                      n_paths=30_000, n_steps=60, seed=42)
    heston_prices.append(h_res['call'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

moneyness_plot = strike_range / S0

ax = axes[0]
ax.plot(moneyness_plot, bs_prices, 'o-', color='#1B3A6B', linewidth=2, label='Black-Scholes')
ax.plot(moneyness_plot, mjd_prices, 's-', color='#C8521A', linewidth=2, label='Merton JD')
ax.plot(moneyness_plot, heston_prices, '^-', color='#2E8B57', linewidth=2, label='Heston')
ax.axvline(1.0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Moneyness (K/S₀)', fontsize=12)
ax.set_ylabel('Call Price (₹)', fontsize=12)
ax.set_title('European Call Prices Across Models', fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Price difference from BS
ax = axes[1]
ax.plot(moneyness_plot, np.array(mjd_prices) - np.array(bs_prices), 's-', 
        color='#C8521A', linewidth=2, label='MJD − BS')
ax.plot(moneyness_plot, np.array(heston_prices) - np.array(bs_prices), '^-', 
        color='#2E8B57', linewidth=2, label='Heston − BS')
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(1.0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Moneyness (K/S₀)', fontsize=12)
ax.set_ylabel('Price Difference from BS (₹)', fontsize=12)
ax.set_title('Pricing Deviation from Black-Scholes', fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}17_option_prices_all_models.png')
plt.show()

---
## 9. Summary Statistics & Model Comparison Table

In [ ]:
# Excess Kurtosis comparison
emp_kurt = float(log_returns.kurtosis())
bs_kurt = 0.0  # Gaussian: excess kurtosis = 0
mjd_kurt_analytical = 3 * lam_m * (muj_m**2 + sigj_m**2)**2 / (sig_m**2 + lam_m*(muj_m**2 + sigj_m**2))**2

print('=== Excess Kurtosis Comparison ===')
print(f'  Empirical NIFTY:    {emp_kurt:.4f}')
print(f'  BS (Gaussian):      {bs_kurt:.4f}')
print(f'  MJD (analytical):   {mjd_kurt_analytical:.4f}')

print('\n=== Complete Model Comparison ===')
comparison = pd.DataFrame({
    'Metric': ['Parameters', 'Log-Likelihood', 'AIC', 'BIC', 
               'ATM Call (₹)', 'ATM Put (₹)', 'Excess Kurtosis', 'Volatility Smile'],
    'Black-Scholes': [2, f'{ll_bs:.1f}', f'{aic_bs:.1f}', f'{bic_bs:.1f}', 
                      f'{call_bs:.2f}', f'{put_bs:.2f}', '0 (by definition)', 'No (flat IV)'],
    'Merton JD': [5, f'{ll_mjd:.1f}', f'{aic_mjd:.1f}', f'{bic_mjd:.1f}', 
                  f'{mjd_call:.2f}', f'{mjd_put:.2f}', f'{mjd_kurt_analytical:.4f}', 'Partial (jump skew)'],
    'Heston': [5, 'N/A (proposal)', 'N/A', 'N/A',
              f'{heston_result["call"]:.2f}', f'{heston_result["put"]:.2f}', 'Stochastic', 'Full smile']
})
print(comparison.to_string(index=False))

# Save for PPTX
comparison.to_csv(f'{FIGURES_DIR}model_comparison.csv', index=False)

In [ ]:
# Final summary figure
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Model hierarchy
ax = axes[0, 0]
ax.barh(['BS', 'MJD', 'Heston'], [2, 5, 5], color=['#1B3A6B', '#C8521A', '#2E8B57'], alpha=0.8)
ax.set_xlabel('Number of Parameters')
ax.set_title('Model Complexity', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# 2. Log-likelihood
ax = axes[0, 1]
bars = ax.bar(['BS', 'MJD'], [ll_bs, ll_mjd], color=['#1B3A6B', '#C8521A'], alpha=0.8, edgecolor='black')
for bar, val in zip(bars, [ll_bs, ll_mjd]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'{val:.0f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
ax.set_title('Log-Likelihood', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# 3. ATM Prices
ax = axes[0, 2]
x_pos = np.arange(3)
call_prices_all = [call_bs, mjd_call, heston_result['call']]
put_prices_all = [put_bs, mjd_put, heston_result['put']]
width = 0.35
ax.bar(x_pos - width/2, call_prices_all, width, color=['#1B3A6B', '#C8521A', '#2E8B57'], 
       alpha=0.8, label='Call')
ax.bar(x_pos + width/2, put_prices_all, width, color=['#1B3A6B', '#C8521A', '#2E8B57'], 
       alpha=0.4, label='Put', hatch='//')
ax.set_xticks(x_pos)
ax.set_xticklabels(['BS', 'MJD', 'Heston'])
ax.set_title('ATM Option Prices (₹)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 4. Return distribution overlay
ax = axes[1, 0]
x_kde2 = np.linspace(-0.06, 0.06, 500)
ax.plot(x_kde2, kde_emp(x_kde2), 'k-', linewidth=2.5, label='Empirical')
ax.plot(x_kde2, kde_bs(x_kde2), '--', color='#1B3A6B', linewidth=2, label='BS')
ax.plot(x_kde2, kde_mjd(x_kde2), '--', color='#C8521A', linewidth=2, label='MJD')
ax.set_title('Return Densities', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 5. Excess kurtosis
ax = axes[1, 1]
bars = ax.bar(['Empirical', 'BS', 'MJD'], [emp_kurt, bs_kurt, mjd_kurt_analytical],
              color=['gray', '#1B3A6B', '#C8521A'], alpha=0.8, edgecolor='black')
for bar, val in zip(bars, [emp_kurt, bs_kurt, mjd_kurt_analytical]):
    ax.text(bar.get_x() + bar.get_width()/2., max(bar.get_height(), 0) + 0.1,
            f'{val:.2f}', ha='center', fontweight='bold', fontsize=10)
ax.set_title('Excess Kurtosis', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# 6. LRT significance
ax = axes[1, 2]
ax.bar(['LRT Statistic'], [LRT], color='#C8521A', alpha=0.8, edgecolor='black')
ax.axhline(chi2.ppf(0.99, 3), color='red', linestyle='--', label=f'χ²(3) 1% critical = {chi2.ppf(0.99,3):.1f}')
ax.axhline(chi2.ppf(0.95, 3), color='orange', linestyle='--', label=f'χ²(3) 5% critical = {chi2.ppf(0.95,3):.1f}')
ax.text(0, LRT + 2, f'LRT = {LRT:.1f}', ha='center', fontweight='bold', fontsize=12)
ax.set_title('Likelihood Ratio Test\nBS vs MJD', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Stochastic Option Pricing Models — Comprehensive Comparison', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}18_summary_comparison.png')
plt.show()

print('\n✓ All figures saved to figures/ directory')
print('\nFigures generated:')
import os
for f in sorted(os.listdir(FIGURES_DIR)):
    if f.endswith('.png'):
        print(f'  {f}')